## 1. Imports and config

In [1]:
from pathlib import Path 
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split, Subset
from torchvision import transforms, datasets

from src.config import DATASET_STR
DATASET_PATH = Path(DATASET_STR)

IMAGE_SIZE = 128
BATCH_SIZE = 16
EPOCHS = 10


## 2. Preprocessing


### 2.1. Pipelines
 

Pipelines for:
- transforming the images into tensors for the model;
- resizing images;
- image augmentation (just for training dataset) 
- normalization of pixel values (from 0.0 to 1.0 roughly to values from -1 to 1) with this formula: new_value = (old_value - mean) / std

In [2]:
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5]
    )
])

test_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5]
    )
])

### 2.2. Load train and test datasets

In [3]:
full_train_dataset = datasets.ImageFolder(
    DATASET_PATH,
    transform=train_transform
)

full_test_dataset = datasets.ImageFolder(
    DATASET_PATH,
    transform=test_transform
)

### 2.3. Random split 80:20

In [4]:
dataset_size = len(full_train_dataset)
train_size = int(0.8 * dataset_size)
test_size = dataset_size - train_size

train_subset, test_subset = random_split(
    range(dataset_size),
    [train_size, test_size],
    generator=torch.Generator().manual_seed(67)
)

Final datasets

In [5]:
train_dataset = Subset(
    full_train_dataset,
    train_subset.indices
)
test_dataset = Subset(
    full_test_dataset,
    test_subset.indices
)

### 2.4. Data loaders

Give out data to the model in this format: [16, 3, 128, 128]
- 16 = batch size
- 3 = RGB channels
- 128 = image width and height

In [6]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

## 3. CNN model

### 3.1. First convolution layer

3.1.1. Convolution
- 3 RGB channels
- 16 feature maps outputted
- 3x3 filter size
- padding for preserving image size

1 image -> 16 feature maps

In [7]:
conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)

3.1.2. ReLU activation function

Keeps only positive outputs in the feature map's pixels and negatives default to 0

In [8]:
relu = nn.ReLU()

3.1.3. Pooling

Shrinks feature maps by taking 2x2 pixel blocks and choosing the biggest value.

128 x 128 feature map -> 64 x 64 feature map     

In [9]:
pool = nn.MaxPool2d(2)

### 3.2. Second convolution layer

- 16 feature maps per image
- 32 output feature maps
- etc.

For the activation we use relu again and we do pooling this layer also

1 16-layered (feature maps) image -> 32 feature maps (32 x 32 res)

In [10]:
conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)

### 3.3. Third convolution layer

The same as last, multiplied feature map output.

Relu and pooling again.

1 32-layered image -> 64 feature maps (16 x 16 res)

In [11]:
conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)

### 3.4. Flattening

Matrix -> vector for The ANN part

In [12]:
flatten = nn.Flatten()

### 3.5. Hidden linear layer

Takes the 64 feature maps of 16 by 16 res per image and creates 128 neurons.

I use ReLU activation again.

** I'll add more layers when I get more data.

In [13]:
layer1 = nn.Linear(64 * 16 * 16, 128)

### 3.6. Output layer

Sigmoid activation is already part of cross-entropy function so no activation here.

In [14]:
output_layer = nn.Linear(128, 2)

### 3.7. CNN model sequence

In [15]:
model = nn.Sequential(
    conv1, relu, pool,      # layer 1

    conv2, relu, pool,      # layer 2

    conv3, relu, pool,      # layer 3

    flatten,

    layer1, relu,           # ann layer 1

    output_layer
)

## 4. Training

### 4.1. Optimizer

In [16]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001        # wheights update by 0.001
)

### 4.2. Loss function

In [18]:
criterion = nn.CrossEntropyLoss()

### 4.3. Training loop

In [ ]:
for epoch in range(EPOCHS):
    running_loss = 0.0

    for images, labels in train_loader:
        outputs = model(images)

        loss = criterion(outputs, labels)

        optimizer.zero_grad()   # clear previous gradients

        loss.backward()     # back propagation

        optimizer.step()    # update weights

        running_loss += loss.item()

    average_loss = running_loss / len(train_loader)

    print(f"Epoch [{epoch + 1}/{EPOCHS}], Loss: {average_loss:.4f}")

Epoch [1/10], Loss: 0.6255
Epoch [2/10], Loss: 0.6129
Epoch [3/10], Loss: 0.6112
Epoch [4/10], Loss: 0.6062
Epoch [5/10], Loss: 0.6034
Epoch [6/10], Loss: 0.5848
Epoch [7/10], Loss: 0.3736
Epoch [8/10], Loss: 0.1764
Epoch [9/10], Loss: 0.0932
Epoch [10/10], Loss: 0.0790


## 5. Evaluation

### 5.1. Test set evaluation

In [20]:
correct = 0
total = 0

with torch.no_grad():

    for images, labels in test_loader:

        outputs = model(images)

        predictions = outputs.argmax(dim=1)     # chooses class with higher score

        correct += (predictions == labels).sum().item()

        total += labels.size(0)

accuracy = 100 * correct / total

print(f"Validation Accuracy: {accuracy:.2f}%")

Validation Accuracy: 97.08%


### 5.2. Single prediction

In [26]:
image_path = "data/prediction/idle/multiple_fish.png"

image = Image.open(image_path).convert("RGB")

image = test_transform(image)

# Add batch dimension
image = image.unsqueeze(0)

class_names = ["bite", "idle"]

with torch.no_grad():
    output = model(image)

    prediction = output.argmax(dim=1).item()

    predicted_class = class_names[prediction]

    probabilities = torch.softmax(output, dim=1)

print("Predicted class:", predicted_class)
print("Probabilities:", probabilities)

Predicted class: idle
Probabilities: tensor([[1.9638e-04, 9.9980e-01]])


## 6. Save model

In [29]:
torch.save(
    model.state_dict(),
    "models/bite_cnn_beta.pth"
)

- 0 = bite
- 1 = idle